# 두류(콩) 작황 데이터 분석

농촌진흥청 두류 작황 성적 API 데이터 탐색 (2017~2025)

In [ ]:
import json
import pandas as pd
import numpy as np

DATA_DIR = "data"

# 코드 매핑 로드
with open(f"{DATA_DIR}/bean_code_species.json", encoding="utf-8") as f:
    SPECIES = json.load(f)
with open(f"{DATA_DIR}/bean_code_areas.json", encoding="utf-8") as f:
    AREAS = json.load(f)

# 전체 CSV 로드
ctvt = pd.read_csv(f"{DATA_DIR}/bean_cultivation_2017_2025.csv")
r01  = pd.read_csv(f"{DATA_DIR}/bean_round01_2017_2025.csv")
r02  = pd.read_csv(f"{DATA_DIR}/bean_round02_2017_2025.csv")
r03  = pd.read_csv(f"{DATA_DIR}/bean_round03_2017_2025.csv")
r04  = pd.read_csv(f"{DATA_DIR}/bean_round04_2017_2025.csv")

print(f"품종 {len(SPECIES)}종, 지역 {len(AREAS)}곳")
print(f"cultivation: {ctvt.shape}")
print(f"round01:     {r01.shape}")
print(f"round02:     {r02.shape}")
print(f"round03:     {r03.shape}")
print(f"round04:     {r04.shape}")

## 1. 재배 관리 데이터 (cultivation)

In [ ]:
# 컬럼 구성 & 타입 확인
ctvt.info()
print()
ctvt.head(10)

In [ ]:
# 품종명, 지역명 매핑 추가
ctvt["품종명"] = ctvt["spciesCode"].map(SPECIES)
ctvt["지역명"] = ctvt["exprAreaCode"].astype(str).str.zfill(4).map(AREAS)

# 연도 × 지역 분포
print("== 연도별 건수 ==")
print(ctvt["exprYear"].value_counts().sort_index())
print()
print("== 지역별 건수 ==")
print(ctvt["지역명"].value_counts())
print()
print("== 품종별 건수 ==")
print(ctvt["품종명"].value_counts())

In [ ]:
# ctvt1~ctvt8 컬럼의 실제 내용 확인
ctvt_cols = [c for c in ctvt.columns if c.startswith("ctvt")]
print("== ctvt 컬럼별 비결측 수 & 샘플 ==")
for col in ctvt_cols:
    non_null = ctvt[col].dropna()
    non_null = non_null[non_null.astype(str).str.strip() != ""]
    sample = non_null.head(5).tolist()
    print(f"  {col}: {len(non_null)}건  예) {sample}")

## 2. 회차별 데이터 형태 확인

In [ ]:
def summarize_round(df, name):
    """회차 데이터의 exam 컬럼별 비결측 수와 샘플값을 출력"""
    print(f"=== {name} ({df.shape[0]}행 × {df.shape[1]}열) ===")
    exam_cols = sorted([c for c in df.columns if c.startswith("exam")])
    for col in exam_cols:
        non_null = df[col].dropna()
        non_null = non_null[non_null.astype(str).str.strip() != ""]
        sample = non_null.head(3).tolist()
        print(f"  {col:>8s}: {len(non_null):>4d}건  예) {sample}")
    # 공통 컬럼 요약
    print(f"  연도: {sorted(df['exprYear'].unique())}")
    print()

summarize_round(r01, "1회차 - 파종기/출현일")
summarize_round(r02, "2회차 - 초장/분지수/개화기")
summarize_round(r03, "3회차 - 협수/립수/건물중")
summarize_round(r04, "4회차 - 수량/백립중")

In [ ]:
# 각 회차 첫 몇 행 직접 확인
print("== round01 ==")
display(r01.head(5))
print("\n== round02 ==")
display(r02.head(5))
print("\n== round03 ==")
display(r03.head(5))
print("\n== round04 ==")
display(r04.head(5))

## 3. 결측(NaN) 현황

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import platform

if platform.system() == "Darwin":
    mpl.rcParams["font.family"] = "Apple SD Gothic Neo"
elif platform.system() == "Windows":
    mpl.rcParams["font.family"] = "Malgun Gothic"
mpl.rcParams["axes.unicode_minus"] = False

def clean_col(series):
    """공백 문자열도 NaN 처리"""
    return series.replace(r'^\s*$', np.nan, regex=True)

# 각 데이터셋의 주요 컬럼 결측률
datasets = {
    "cultivation\n(재배관리)": ctvt[[c for c in ctvt.columns if c.startswith("ctvt")]],
    "round01\n(파종/출현)": r01[[c for c in r01.columns if c.startswith("exam")]],
    "round02\n(초장/개화)": r02[[c for c in r02.columns if c.startswith("exam")]],
    "round03\n(협수/건물중)": r03[[c for c in r03.columns if c.startswith("exam")]],
    "round04\n(수량/백립중)": r04[[c for c in r04.columns if c.startswith("exam")]],
}

fig, axes = plt.subplots(1, 5, figsize=(16, 4), sharey=False)
fig.suptitle("데이터셋별 주요 컬럼 결측률 (%)", fontsize=13, fontweight="bold")

for ax, (name, df) in zip(axes, datasets.items()):
    pct = df.apply(lambda s: clean_col(s).isna().mean() * 100)
    colors = ["#ef4444" if v > 50 else "#f59e0b" if v > 20 else "#22c55e" for v in pct]
    ax.barh(pct.index, pct.values, color=colors)
    ax.set_xlim(0, 100)
    ax.set_title(name, fontsize=10)
    ax.axvline(50, color="gray", ls="--", lw=0.8, alpha=0.5)

axes[0].set_xlabel("결측률 (%)")
plt.tight_layout()
plt.show()

## 4. 파종일 기준 시각화

- `pajong`: 파종일, `r02.exam1`: 초장(cm), `r02.exam6`: 개화기
- `r04.exam9`: 수량(kg/10a), `r04.exam6`: 협수(개/주)

In [ ]:
# 파종일 분포 (연도별)
r01["pajong_dt"] = pd.to_datetime(r01["pajong"], errors="coerce")
r01["pajong_doy"] = r01["pajong_dt"].dt.dayofyear  # day of year
r01["품종명"] = r01["spciesCode"].map(SPECIES)
r01["지역명"] = r01["exprAreaCode"].astype(str).str.zfill(4).map(AREAS)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# (a) 연도별 파종일 분포
ax = axes[0]
for year in sorted(r01["exprYear"].unique()):
    sub = r01[r01["exprYear"] == year]["pajong_doy"].dropna()
    if len(sub) > 0:
        ax.scatter([year]*len(sub), sub, alpha=0.5, s=30)
ax.set_xlabel("연도")
ax.set_ylabel("파종일 (DOY)")
ax.set_title("연도별 파종일 분포")
ax.grid(True, alpha=0.3)

# (b) 지역별 파종일 boxplot
ax = axes[1]
box_data = []
box_labels = []
for area in sorted(r01["지역명"].dropna().unique()):
    vals = r01[r01["지역명"] == area]["pajong_doy"].dropna()
    if len(vals) > 2:
        box_data.append(vals)
        box_labels.append(area.split("(")[0])  # 짧은 이름
bp = ax.boxplot(box_data, labels=box_labels, patch_artist=True)
for patch in bp["boxes"]:
    patch.set_facecolor("#93c5fd")
ax.set_ylabel("파종일 (DOY)")
ax.set_title("지역별 파종일 분포")
ax.tick_params(axis="x", rotation=30)
ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

In [ ]:
# 파종일 vs 초장 / 수량 (r02, r04를 pajong 기준으로 병합)
r02["pajong_dt"] = pd.to_datetime(r02["pajong"], errors="coerce")
r02["pajong_doy"] = r02["pajong_dt"].dt.dayofyear
r02["초장"] = pd.to_numeric(clean_col(r02["exam1"]), errors="coerce")
r02["품종명"] = r02["spciesCode"].map(SPECIES)

r04["pajong_dt"] = pd.to_datetime(r04["pajong"], errors="coerce")
r04["pajong_doy"] = r04["pajong_dt"].dt.dayofyear
r04["수량"] = pd.to_numeric(clean_col(r04["exam9"]), errors="coerce")   # kg/10a
r04["협수"] = pd.to_numeric(clean_col(r04["exam6"]), errors="coerce")
r04["품종명"] = r04["spciesCode"].map(SPECIES)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# (a) 파종일 vs 초장
ax = axes[0]
mask = r02["pajong_doy"].notna() & r02["초장"].notna()
sc = ax.scatter(r02.loc[mask, "pajong_doy"], r02.loc[mask, "초장"],
                c=r02.loc[mask, "exprYear"], cmap="viridis", alpha=0.6, s=35, edgecolors="w", lw=0.3)
ax.set_xlabel("파종일 (DOY)")
ax.set_ylabel("초장 (cm)")
ax.set_title("파종일 vs 초장 (2회차)")
ax.grid(True, alpha=0.3)
plt.colorbar(sc, ax=ax, label="연도")

# (b) 파종일 vs 수량
ax = axes[1]
mask = r04["pajong_doy"].notna() & r04["수량"].notna()
sc = ax.scatter(r04.loc[mask, "pajong_doy"], r04.loc[mask, "수량"],
                c=r04.loc[mask, "exprYear"], cmap="viridis", alpha=0.6, s=35, edgecolors="w", lw=0.3)
ax.set_xlabel("파종일 (DOY)")
ax.set_ylabel("수량 (kg/10a)")
ax.set_title("파종일 vs 수량 (4회차)")
ax.grid(True, alpha=0.3)
plt.colorbar(sc, ax=ax, label="연도")

# (c) 파종일 vs 협수
ax = axes[2]
mask = r04["pajong_doy"].notna() & r04["협수"].notna()
sc = ax.scatter(r04.loc[mask, "pajong_doy"], r04.loc[mask, "협수"],
                c=r04.loc[mask, "exprYear"], cmap="viridis", alpha=0.6, s=35, edgecolors="w", lw=0.3)
ax.set_xlabel("파종일 (DOY)")
ax.set_ylabel("협수 (개/주)")
ax.set_title("파종일 vs 협수 (4회차)")
ax.grid(True, alpha=0.3)
plt.colorbar(sc, ax=ax, label="연도")

plt.tight_layout()
plt.show()

In [ ]:
# 품종별 수량 비교 (NaN 제거 후)
fig, ax = plt.subplots(figsize=(10, 4))

box_data = []
box_labels = []
for sp in sorted(r04["품종명"].dropna().unique()):
    vals = r04[r04["품종명"] == sp]["수량"].dropna()
    if len(vals) >= 3:
        box_data.append(vals)
        box_labels.append(sp)

bp = ax.boxplot(box_data, labels=box_labels, patch_artist=True, widths=0.6)
colors = plt.cm.Set3(np.linspace(0, 1, len(box_data)))
for patch, c in zip(bp["boxes"], colors):
    patch.set_facecolor(c)

ax.set_ylabel("수량 (kg/10a)")
ax.set_title("품종별 수량 분포 (4회차, NaN 제외)")
ax.tick_params(axis="x", rotation=40)
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()